# MICO Set-up 현황 조회 (PostgreSQL)
PostgreSQL DB에서 Category / SubCategory / Detail 데이터를 DataFrame으로 추출

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd

# PostgreSQL 연결 (SQLAlchemy)
DB_URL = 'postgresql+psycopg2://bonheonkoo@localhost:5432/mico_web'
engine = create_engine(DB_URL)

with engine.connect() as conn:
    print('PostgreSQL 연결 완료:', conn.execute(text('SELECT version()')).scalar())

PostgreSQL 연결 완료: PostgreSQL 14.12 (Homebrew) on aarch64-apple-darwin23.4.0, compiled by Apple clang version 15.0.0 (clang-1500.3.9.4), 64-bit


## 1. 각 테이블 단독 조회

In [2]:
# Category
df_category = pd.read_sql_query("""
    SELECT id, family, product, oper_id, oper_desc, created_at
    FROM setup_mico_category
    ORDER BY family, product, oper_id
""", engine)

print(f'Category: {len(df_category)}건')
df_category

Category: 1건


,id,family,product,oper_id,oper_desc,created_at
0,1,NAND,OP,A908740A,M1 CU CMP,2026-03-24 13:32:16.171983+00:00


In [3]:
# SubCategory
df_subcategory = pd.read_sql_query("""
    SELECT id, category_id, fab, device, recipe_id, maker, created_at
    FROM setup_mico_subcategory
    ORDER BY category_id, fab, device
""", engine)

print(f'SubCategory: {len(df_subcategory)}건')
df_subcategory

SubCategory: 1건


,id,category_id,fab,device,recipe_id,maker,created_at
0,1,1,M11,9C,9C_M1CU_R04.cas,AMAT,2026-03-24 13:32:47.550047+00:00


In [4]:
# Detail
df_detail = pd.read_sql_query("""
    SELECT id, subcategory_id,
           apc_para, thk_para,
           target, pre_target, pre_thk_period,
           rr_para, offset_group, rr_max, rr_period, rr_if,
           rr_weight, rr_count,
           pre_thk_para_itm,
           pre_oper_code,  pre_oper_desc,  pre_oper_para,
           pre_oper_code2, pre_oper_desc2, pre_oper_para2,
           pre_oper_code3, pre_oper_desc3, pre_oper_para3,
           pre_oper_code4, pre_oper_desc4, pre_oper_para4,
           created_at
    FROM setup_mico_detail
    ORDER BY subcategory_id
""", engine)

print(f'Detail: {len(df_detail)}건')
df_detail

Detail: 1건


,id,subcategory_id,apc_para,thk_para,target,pre_target,pre_thk_period,rr_para,offset_group,rr_max,...,pre_oper_code2,pre_oper_desc2,pre_oper_para2,pre_oper_code3,pre_oper_desc3,pre_oper_para3,pre_oper_code4,pre_oper_desc4,pre_oper_para4,created_at
0,1,1,P3,AMAT_ITM_POST_THK1_AVG,700,450,10,pad,N,None,...,A222222B,M1 ETCH,6_processTime,A33333C,M1 TRENCH OX CMP,EBARA_ITM_POST_THK1_AVG,A44444D,M1 BM/SEED CU DEP PRE CLN PRE OCD,T3_AVG,2026-03-24 13:34:47.591578+00:00


## 2. 전체 계층 JOIN — SQL 방식 (Category → SubCategory → Detail)

In [5]:
df_full = pd.read_sql_query("""
    SELECT
        c.family,
        c.product,
        c.oper_id,
        c.oper_desc,
        s.fab,
        s.device,
        s.recipe_id,
        s.maker,
        d.apc_para,
        d.thk_para,
        d.target,
        d.pre_target,
        d.pre_thk_period,
        d.rr_para,
        d.offset_group,
        d.rr_max,
        d.rr_period,
        d.rr_if,
        d.rr_weight,
        d.rr_count,
        d.pre_thk_para_itm,
        d.pre_oper_code,  d.pre_oper_desc,  d.pre_oper_para,
        d.pre_oper_code2, d.pre_oper_desc2, d.pre_oper_para2,
        d.pre_oper_code3, d.pre_oper_desc3, d.pre_oper_para3,
        d.pre_oper_code4, d.pre_oper_desc4, d.pre_oper_para4
    FROM setup_mico_detail      AS d
    JOIN setup_mico_subcategory AS s ON d.subcategory_id = s.id
    JOIN setup_mico_category    AS c ON s.category_id   = c.id
    ORDER BY c.family, c.product, c.oper_id, s.fab, s.device, s.recipe_id
""", engine)

print(f'전체 Set-up: {len(df_full)}건')
df_full

전체 Set-up: 1건


,family,product,oper_id,oper_desc,fab,device,recipe_id,maker,apc_para,thk_para,...,pre_oper_para,pre_oper_code2,pre_oper_desc2,pre_oper_para2,pre_oper_code3,pre_oper_desc3,pre_oper_para3,pre_oper_code4,pre_oper_desc4,pre_oper_para4
0,NAND,OP,A908740A,M1 CU CMP,M11,9C,9C_M1CU_R04.cas,AMAT,P3,AMAT_ITM_POST_THK1_AVG,...,Chamber,A222222B,M1 ETCH,6_processTime,A33333C,M1 TRENCH OX CMP,EBARA_ITM_POST_THK1_AVG,A44444D,M1 BM/SEED CU DEP PRE CLN PRE OCD,T3_AVG


## 3. 전체 계층 JOIN — pandas merge 방식

In [6]:
df_merge = (
    df_detail
    .merge(
        df_subcategory.rename(columns={'id': 'subcategory_id'}),
        on='subcategory_id',
        suffixes=('', '_sub')
    )
    .merge(
        df_category.rename(columns={'id': 'category_id'}),
        on='category_id',
        suffixes=('', '_cat')
    )
)

# 컬럼 순서 정리
cols = [
    'family', 'product', 'oper_id', 'oper_desc',
    'fab', 'device', 'recipe_id', 'maker',
    'apc_para', 'thk_para', 'target', 'pre_target', 'pre_thk_period',
    'rr_para', 'offset_group', 'rr_max', 'rr_period', 'rr_if',
    'rr_weight', 'rr_count',
    'pre_thk_para_itm',
    'pre_oper_code',  'pre_oper_desc',  'pre_oper_para',
    'pre_oper_code2', 'pre_oper_desc2', 'pre_oper_para2',
    'pre_oper_code3', 'pre_oper_desc3', 'pre_oper_para3',
    'pre_oper_code4', 'pre_oper_desc4', 'pre_oper_para4',
]
df_merge = (
    df_merge[cols]
    .sort_values(['family', 'product', 'oper_id', 'fab', 'device'])
    .reset_index(drop=True)
)

print(f'전체 Set-up: {len(df_merge)}건')
df_merge

전체 Set-up: 1건


,family,product,oper_id,oper_desc,fab,device,recipe_id,maker,apc_para,thk_para,...,pre_oper_para,pre_oper_code2,pre_oper_desc2,pre_oper_para2,pre_oper_code3,pre_oper_desc3,pre_oper_para3,pre_oper_code4,pre_oper_desc4,pre_oper_para4
0,NAND,OP,A908740A,M1 CU CMP,M11,9C,9C_M1CU_R04.cas,AMAT,P3,AMAT_ITM_POST_THK1_AVG,...,Chamber,A222222B,M1 ETCH,6_processTime,A33333C,M1 TRENCH OX CMP,EBARA_ITM_POST_THK1_AVG,A44444D,M1 BM/SEED CU DEP PRE CLN PRE OCD,T3_AVG


## 4. 조건 필터링 예시

In [7]:
# Family로 필터
TARGET_FAMILY = 'NAND'   # 'NAND' or 'DRAM'

df_by_family = df_full[df_full['family'] == TARGET_FAMILY].reset_index(drop=True)
print(f'{TARGET_FAMILY} → {len(df_by_family)}건')
df_by_family

NAND → 1건


,family,product,oper_id,oper_desc,fab,device,recipe_id,maker,apc_para,thk_para,...,pre_oper_para,pre_oper_code2,pre_oper_desc2,pre_oper_para2,pre_oper_code3,pre_oper_desc3,pre_oper_para3,pre_oper_code4,pre_oper_desc4,pre_oper_para4
0,NAND,OP,A908740A,M1 CU CMP,M11,9C,9C_M1CU_R04.cas,AMAT,P3,AMAT_ITM_POST_THK1_AVG,...,Chamber,A222222B,M1 ETCH,6_processTime,A33333C,M1 TRENCH OX CMP,EBARA_ITM_POST_THK1_AVG,A44444D,M1 BM/SEED CU DEP PRE CLN PRE OCD,T3_AVG


In [8]:
# Product + Oper ID로 필터
TARGET_PRODUCT = 'LC'
TARGET_OPER_ID = 'A908740A'

df_filtered = df_full[
    (df_full['product'] == TARGET_PRODUCT) &
    (df_full['oper_id'] == TARGET_OPER_ID)
].reset_index(drop=True)

print(f'{TARGET_PRODUCT} / {TARGET_OPER_ID} → {len(df_filtered)}건')
df_filtered

LC / A908740A → 0건


,family,product,oper_id,oper_desc,fab,device,recipe_id,maker,apc_para,thk_para,...,pre_oper_para,pre_oper_code2,pre_oper_desc2,pre_oper_para2,pre_oper_code3,pre_oper_desc3,pre_oper_para3,pre_oper_code4,pre_oper_desc4,pre_oper_para4


In [9]:
# SQL 파라미터 바인딩으로 직접 필터링 (대용량 데이터에 유리)
TARGET_PRODUCT = 'LC'

with engine.connect() as conn:
    df_sql_filtered = pd.read_sql_query(
        text("""
            SELECT
                c.family, c.product, c.oper_id, c.oper_desc,
                s.fab, s.device, s.recipe_id, s.maker,
                d.apc_para, d.thk_para, d.target, d.pre_target, d.pre_thk_period,
                d.rr_para, d.offset_group, d.rr_max, d.rr_period, d.rr_if
            FROM setup_mico_detail      AS d
            JOIN setup_mico_subcategory AS s ON d.subcategory_id = s.id
            JOIN setup_mico_category    AS c ON s.category_id   = c.id
            WHERE c.product = :product
            ORDER BY c.oper_id, s.fab, s.device
        """),
        conn,
        params={'product': TARGET_PRODUCT}
    )

print(f'{TARGET_PRODUCT} → {len(df_sql_filtered)}건')
df_sql_filtered

LC → 0건


,family,product,oper_id,oper_desc,fab,device,recipe_id,maker,apc_para,thk_para,target,pre_target,pre_thk_period,rr_para,offset_group,rr_max,rr_period,rr_if


## 5. 요약 통계

In [10]:
print('=== Family / Product별 SubCategory / Detail 수 ===')
summary = (
    df_full
    .groupby(['family', 'product', 'oper_id', 'oper_desc'])
    .agg(
        subcategory_count=('recipe_id', 'nunique'),
        detail_count=('apc_para', 'count')
    )
    .reset_index()
)
summary

=== Family / Product별 SubCategory / Detail 수 ===


,family,product,oper_id,oper_desc,subcategory_count,detail_count
0,NAND,OP,A908740A,M1 CU CMP,1,1


In [11]:
print('=== Maker별 Detail 수 ===')
(
    df_full
    .groupby(['family', 'maker'])
    .agg(detail_count=('apc_para', 'count'))
    .reset_index()
)

=== Maker별 Detail 수 ===


,family,maker,detail_count
0,NAND,AMAT,1


In [12]:
# 엔진 종료
engine.dispose()
print('PostgreSQL 연결 종료')

PostgreSQL 연결 종료
